1. Crear el DataFrame inicial

In [1]:
# Paso 1: importar librerías básicas
import pandas as pd
import numpy as np


In [2]:
# Paso 2: definir los datos en un diccionario
data = {
    'ID': [1, 2, 3, 4],
    'Edad': [25, 45, 30, 40],
    'Ciudad': ['Madrid', 'Sevilla', 'Madrid', 'Barcelona'],
    'Salario': [30000, 50000, np.nan, 40000]
}


In [3]:
# Paso 3: crear el DataFrame
df = pd.DataFrame(data)

# Paso 4: mostrar los datos originales
print("Datos originales:\n", df)


Datos originales:
    ID  Edad     Ciudad  Salario
0   1    25     Madrid  30000.0
1   2    45    Sevilla  50000.0
2   3    30     Madrid      NaN
3   4    40  Barcelona  40000.0


Interpretación: aquí solo cargamos los datos tal como vienen en el enunciado, con un salario faltante (NaN) en el ID 3.

2. Imputar el salario faltante con la media

In [4]:
# Paso 5: importar el imputador simple
from sklearn.impute import SimpleImputer


In [5]:
# Paso 6: crear el imputador con estrategia 'mean' (media)
imputer = SimpleImputer(strategy='mean')


In [6]:
# Paso 7: ajustar el imputador a la columna 'Salario' y transformar
df['Salario'] = imputer.fit_transform(df[['Salario']])


In [7]:
# Paso 8: verificar los datos con el salario imputado
print("\nDatos con salario imputado:\n", df)



Datos con salario imputado:
    ID  Edad     Ciudad  Salario
0   1    25     Madrid  30000.0
1   2    45    Sevilla  50000.0
2   3    30     Madrid  40000.0
3   4    40  Barcelona  40000.0


Interpretación: la media se calcula como 
(30000+50000+40000)/3=40000
(30000+50000+40000)/3=40000, por lo que el valor faltante se reemplaza por 40000.

3. Codificar la columna Ciudad (Label Encoding)

In [8]:
# Paso 9: importar LabelEncoder
from sklearn.preprocessing import LabelEncoder


In [9]:
# Paso 10: crear el objeto codificador
le = LabelEncoder()


In [10]:
# Paso 11: ajustar el codificador y transformar la columna 'Ciudad'
df['Ciudad_Label'] = le.fit_transform(df['Ciudad'])


Interpretación: cada ciudad se reemplaza por un número entero (por ejemplo, Barcelona → 0, Madrid → 1, Sevilla → 2, según el orden interno alfabético). Esto sirve para modelos que solo aceptan números, pero conserva un orden artificial que no es real.
​

4. One-Hot Encoding sobre Ciudad

In [11]:
# Paso 13: importar OneHotEncoder
from sklearn.preprocessing import OneHotEncoder


In [12]:
from sklearn.preprocessing import OneHotEncoder

# Antes (da error)
# ohe = OneHotEncoder(sparse=False)

# Ahora (versión nueva de scikit-learn)
ohe = OneHotEncoder(sparse_output=False)

In [13]:
# Paso 15: ajustar y transformar la columna 'Ciudad'
ciudad_encoded = ohe.fit_transform(df[['Ciudad']])


In [14]:
# Paso 16: convertir la matriz resultante a DataFrame con nombres de columnas
df_ohe = pd.DataFrame(
    ciudad_encoded,
    columns=ohe.get_feature_names_out(['Ciudad'])
)

# Paso 17: mostrar el resultado
print("\nOne-Hot Encoding:\n", df_ohe)



One-Hot Encoding:
    Ciudad_Barcelona  Ciudad_Madrid  Ciudad_Sevilla
0               0.0            1.0             0.0
1               0.0            0.0             1.0
2               0.0            1.0             0.0
3               1.0            0.0             0.0


Interpretación: ahora cada ciudad se representa con columnas binarias (0/1), por ejemplo una fila de Madrid tendrá Ciudad_Madrid = 1, Ciudad_Barcelona = 0, Ciudad_Sevilla = 0. Esta representación no introduce orden entre categorías.


5. Variables Dummy con get_dummies

In [15]:
# Paso 18: crear variables dummy a partir de 'Ciudad'
df_dummy = pd.get_dummies(df['Ciudad'], drop_first=True)


In [16]:
# Paso 19: mostrar las dummies
print("\nVariables Dummy:\n", df_dummy)



Variables Dummy:
    Madrid  Sevilla
0    True    False
1   False     True
2    True    False
3   False    False


Interpretación: drop_first=True elimina una de las categorías para evitar multicolinealidad (la categoría eliminada queda implícita cuando todas las demás son 0). Por ejemplo, si se elimina Barcelona, una fila con todas las dummies en 0 representa “Barcelona”.

6. Escalamiento Min-Max en Edad y Salario

In [17]:
# Paso 20: importar el escalador Min-Max y StandardScaler
from sklearn.preprocessing import MinMaxScaler, StandardScaler


In [18]:
# Paso 21: crear el escalador Min-Max
scaler_minmax = MinMaxScaler()


In [19]:
# Paso 22: ajustar el escalador a Edad y Salario y transformar
df[['Edad_MinMax', 'Salario_MinMax']] = scaler_minmax.fit_transform(
    df[['Edad', 'Salario']]
)


In [20]:
# Paso 23: mostrar los datos escalados con Min-Max
print("\nMin-Max Scaling:\n", df[['Edad_MinMax', 'Salario_MinMax']])



Min-Max Scaling:
    Edad_MinMax  Salario_MinMax
0         0.00             0.0
1         1.00             1.0
2         0.25             0.5
3         0.75             0.5


Interpretación: la fórmula es \(x' = \frac{x - x_{\min}}{x_{\max} - x_{\min}}\). En este caso los valores quedan entre 0 y 1; por ejemplo, si la **edad** mínima es 25 y la máxima 45, una edad de 35 quedaría en el centro del rango (aprox. 0.5).

7. Estandarización (Z-Score) en Edad y Salario

In [21]:
# Paso 24: crear el escalador estándar (Z-Score)
scaler_std = StandardScaler()


In [22]:
# Paso 25: ajustar y transformar Edad y Salario
df[['Edad_Std', 'Salario_Std']] = scaler_std.fit_transform(
    df[['Edad', 'Salario']]
)


In [23]:
# Paso 26: mostrar los datos estandarizados
print("\nEstandarización:\n", df[['Edad_Std', 'Salario_Std']])



Estandarización:
    Edad_Std  Salario_Std
0 -1.264911    -1.414214
1  1.264911     1.414214
2 -0.632456     0.000000
3  0.632456     0.000000


Interpretación: la estandarización usa z=(x−μ)/σz=(x−μ)/σ, donde μμ es la media y σσ la desviación estándar. Los nuevos valores tienen media 0 y desviación estándar 1, lo que hace comparables variables con escalas muy diferentes.

8. Código completo tipo notebook

In [24]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, MinMaxScaler, StandardScaler
from sklearn.impute import SimpleImputer

# Datos
data = {
    'ID': [1, 2, 3, 4],
    'Edad': [25, 45, 30, 40],
    'Ciudad': ['Madrid', 'Sevilla', 'Madrid', 'Barcelona'],
    'Salario': [30000, 50000, np.nan, 40000]
}

df = pd.DataFrame(data)
print("Datos originales:\n", df)

# 1. Imputar salario con la media
imputer = SimpleImputer(strategy='mean')
df['Salario'] = imputer.fit_transform(df[['Salario']])
print("\nDatos con salario imputado:\n", df)

# 2. Label Encoding
le = LabelEncoder()
df['Ciudad_Label'] = le.fit_transform(df['Ciudad'])
print("\nLabel Encoding:\n", df[['Ciudad', 'Ciudad_Label']])

# 3. One-Hot Encoding
ohe = OneHotEncoder(sparse_output=False)
#ohe = OneHotEncoder(sparse=False)
ciudad_encoded = ohe.fit_transform(df[['Ciudad']])
df_ohe = pd.DataFrame(ciudad_encoded, columns=ohe.get_feature_names_out(['Ciudad']))
print("\nOne-Hot Encoding:\n", df_ohe)

# 4. Variables Dummy
df_dummy = pd.get_dummies(df['Ciudad'], drop_first=True)
print("\nVariables Dummy:\n", df_dummy)

# 5. Min-Max Scaling
scaler_minmax = MinMaxScaler()
df[['Edad_MinMax', 'Salario_MinMax']] = scaler_minmax.fit_transform(df[['Edad', 'Salario']])
print("\nMin-Max Scaling:\n", df[['Edad_MinMax', 'Salario_MinMax']])

# 6. Estandarización (Z-Score)
scaler_std = StandardScaler()
df[['Edad_Std', 'Salario_Std']] = scaler_std.fit_transform(df[['Edad', 'Salario']])
print("\nEstandarización:\n", df[['Edad_Std', 'Salario_Std']])


Datos originales:
    ID  Edad     Ciudad  Salario
0   1    25     Madrid  30000.0
1   2    45    Sevilla  50000.0
2   3    30     Madrid      NaN
3   4    40  Barcelona  40000.0

Datos con salario imputado:
    ID  Edad     Ciudad  Salario
0   1    25     Madrid  30000.0
1   2    45    Sevilla  50000.0
2   3    30     Madrid  40000.0
3   4    40  Barcelona  40000.0

Label Encoding:
       Ciudad  Ciudad_Label
0     Madrid             1
1    Sevilla             2
2     Madrid             1
3  Barcelona             0

One-Hot Encoding:
    Ciudad_Barcelona  Ciudad_Madrid  Ciudad_Sevilla
0               0.0            1.0             0.0
1               0.0            0.0             1.0
2               0.0            1.0             0.0
3               1.0            0.0             0.0

Variables Dummy:
    Madrid  Sevilla
0    True    False
1   False     True
2    True    False
3   False    False

Min-Max Scaling:
    Edad_MinMax  Salario_MinMax
0         0.00             0.0
1       